### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.1:8b")

In [13]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [14]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T15:11:08.6095488Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4183695200, 'load_duration': 3698658800, 'prompt_eval_count': 159, 'prompt_eval_duration': 122831000, 'eval_count': 17, 'eval_duration': 354789000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'} id='lc_run--01a043c6-4ce8-7b03-8e5b-47a4a320f5f7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'bbaae7db-bbab-4b2a-84c4-732ec76628ad', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 159, 'output_tokens': 17, 'total_tokens': 176}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [15]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

In [16]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-08-27T15:11:47.7603812Z', 'done': True, 'done_reason': 'stop', 'total_duration': 667078400, 'load_duration': 9592800, 'prompt_eval_count': 158, 'prompt_eval_duration': 301779000, 'eval_count': 17, 'eval_duration': 349193000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a043c6-f394-71f0-acd0-6c588a3d4670-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '730628aa-64e6-4a90-b35a-8359a26d40f8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 17, 'total_tokens': 175}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='730628aa-64e6-4a90-b35a-8359a26d40f8')]